In [1]:
import urllib.request

url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
file_name = "input.txt"

urllib.request.urlretrieve(url, file_name)
print("Dataset downloaded successfully!")

Dataset downloaded successfully!


In [2]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

print("length of dataset in characters:", len(text))
print(text[:250])

length of dataset in characters: 1115394
First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.



In [3]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(''.join(chars))
print(vocab_size)

stoi = { ch: i for i, ch in enumerate(chars) }
itos = { i: ch for i, ch in enumerate(chars) }

encode = lambda s: [stoi[c] for c in s]
decode = lambda l: ''.join(itos[i] for i in l)

print(encode("hii there"))
print(decode(encode("hii there")))


 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz
65
[46, 47, 47, 1, 58, 46, 43, 56, 43]
hii there


In [4]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)

n = int(0.9 * len(data))
train_data = data[:n]
val_data = data[n:]

print("training tokens:", len(train_data))
print("validation tokens:", len(val_data))

torch.Size([1115394]) torch.int64
training tokens: 1003854
validation tokens: 111540


In [5]:
block_size = 8
batch_size = 4

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

xb, yb = get_batch('train')
print('inputs:')
print(xb.shape)
print(xb)
print('targets:')
print(yb.shape)
print(yb)

inputs:
torch.Size([4, 8])
tensor([[39, 52, 42,  1, 39,  1, 25, 53],
        [ 1, 46, 39, 52, 42, 50, 47, 52],
        [51, 53, 57, 58,  1, 61, 39, 56],
        [46,  1, 46, 47, 51,  8,  0,  0]])
targets:
torch.Size([4, 8])
tensor([[52, 42,  1, 39,  1, 25, 53, 52],
        [46, 39, 52, 42, 50, 47, 52, 45],
        [53, 57, 58,  1, 61, 39, 56, 50],
        [ 1, 46, 47, 51,  8,  0,  0, 32]])


In [6]:
import torch.nn as nn
from torch.nn import functional as F

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx)  # (B, T, vocab_size)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            logits, loss = self(idx)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(loss)

tensor(4.9038, grad_fn=<NllLossBackward0>)


In [7]:
optimizer = torch.optim.AdamW(m.parameters(), lr=1e-3)

for steps in range(10000):
    xb, yb = get_batch('train')

    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

2.5310356616973877


In [8]:
idx = torch.zeros((1, 1), dtype=torch.long)
print(decode(m.generate(idx, max_new_tokens=300)[0].tolist()))


TUKEd'lf s mewh wesald st P n!
Whane; hed anty, aty th;-r t-ge:
Hougr priso musof yos beqOWa theathis, s t MELE
cathivQTRysk mete;
INameBOYXr;j'd'-r:
YFL
E che tomal.

Mkng; athenate HOrifos yioreaend wist n mybe g wig, by g, ats t ato ash, st t chadserESAMEpis towly, fordit u ISiccor'tatth
Tre fuke


In [9]:
torch.manual_seed(1337)
B, T, C = 4, 8, 2   # batch, time (sequence length), channels
x = torch.randn(B, T, C)

# a lower-triangular matrix of 1s...
tril = torch.tril(torch.ones(T, T))
# ...normalized so each row sums to 1, i.e. each row is an average
wei = tril / tril.sum(1, keepdim=True)

# (T,T) @ (B,T,C) --broadcasts--> (B,T,C)
xbow = wei @ x

print(wei)
print(xbow[0])

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5000, 0.5000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3333, 0.3333, 0.3333, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2500, 0.2500, 0.2500, 0.2500, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2000, 0.2000, 0.2000, 0.2000, 0.2000, 0.0000, 0.0000, 0.0000],
        [0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.1667, 0.0000, 0.0000],
        [0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.1429, 0.0000],
        [0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250, 0.1250]])
tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])


In [10]:
head_size = 16
key   = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

k = key(x)      # (B, T, head_size)  "what do I contain"
q = query(x)    # (B, T, head_size)  "what am I looking for"

wei = q @ k.transpose(-2, -1) * head_size**-0.5   # (B, T, T)

tril = torch.tril(torch.ones(T, T))
wei = wei.masked_fill(tril == 0, float('-inf'))
wei = F.softmax(wei, dim=-1)

v = value(x)
out = wei @ v

print(wei[0])
print(out.shape)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5150, 0.4850, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3342, 0.2973, 0.3686, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2330, 0.1956, 0.2701, 0.3013, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2028, 0.2231, 0.1988, 0.1965, 0.1788, 0.0000, 0.0000, 0.0000],
        [0.1514, 0.1827, 0.1263, 0.1103, 0.1483, 0.2809, 0.0000, 0.0000],
        [0.1438, 0.1328, 0.1517, 0.1577, 0.1505, 0.1194, 0.1442, 0.0000],
        [0.1049, 0.1558, 0.0818, 0.0683, 0.0813, 0.2506, 0.1041, 0.1533]],
       grad_fn=<SelectBackward0>)
torch.Size([4, 8, 16])


In [11]:
n_embd = 32
block_size = 8
dropout = 0.0

class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()
        self.key   = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        B, T, C = x.shape
        k, q = self.key(x), self.query(x)
        wei = q @ k.transpose(-2, -1) * C**-0.5
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))
        wei = F.softmax(wei, dim=-1)
        wei = self.dropout(wei)
        return wei @ self.value(x)

class MultiHeadAttention(nn.Module):
    """ several heads of self-attention in parallel """
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.dropout(self.proj(out))

print("Head and MultiHeadAttention classes defined successfully!")

Head and MultiHeadAttention classes defined successfully!


In [12]:
class FeedForward(nn.Module):
    """ a simple linear layer followed by a non-linearity """
    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)

print("FeedForward class defined successfully!")

FeedForward class defined successfully!


In [13]:
class Block(nn.Module):
    """ Transformer block: communication, then computation """
    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x))     # residual around attention
        x = x + self.ffwd(self.ln2(x))   # residual around feed-forward
        return x

print("Block class defined successfully!")

Block class defined successfully!


In [14]:
class GPTLanguageModel(nn.Module):
    def __init__(self, vocab_size, n_embd, block_size, n_head, n_layer):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(*[Block(n_embd, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding_table(idx)                      # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T))       # (T,C)
        x = tok_emb + pos_emb
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)                                       # (B,T,vocab_size)

        loss = None
        if targets is not None:
            B, T, C = logits.shape
            loss = F.cross_entropy(logits.view(B*T, C), targets.view(B*T))
        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

print("GPTLanguageModel class defined successfully!")

GPTLanguageModel class defined successfully!


In [15]:
# Hyperparameters
n_embd = 64
n_head = 4
n_layer = 4
dropout = 0.0
block_size = 32
batch_size = 16
learning_rate = 3e-4
max_iters = 3000
eval_interval = 300

model = GPTLanguageModel(vocab_size, n_embd, block_size, n_head, n_layer)
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(200)
        for k in range(200):
            X, Y = get_batch(split)
            _, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")

    xb, yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print("Training finished! Final loss:", loss.item())

step 0: train loss 4.3500, val loss 4.3502
step 300: train loss 2.6348, val loss 2.6361
step 600: train loss 2.4713, val loss 2.4808
step 900: train loss 2.3817, val loss 2.3951
step 1200: train loss 2.3063, val loss 2.3226
step 1500: train loss 2.2581, val loss 2.2660
step 1800: train loss 2.2007, val loss 2.2249
step 2100: train loss 2.1568, val loss 2.1772
step 2400: train loss 2.1221, val loss 2.1516
step 2700: train loss 2.0857, val loss 2.1318
Training finished! Final loss: 2.0492911338806152


In [16]:
context = torch.zeros((1, 1), dtype=torch.long)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


ROREOFETIO:
Lo, Gevely mave is
I leer and do glist en thee my I hart mither.

Sost surce bore thear kiffu'd?

BEESAS:
I, my Catier angais hey rarnce; Pavunt be the sraioucte, whou is y roury affer,
And heer hime of in abe riptw eattasel' tie a larest's'lors shearyns,
And Gepraimps p coy blad ecup, Cour buth's with in mavediss!
And As tigand you
's srechir an ther reom an till for ochan:
And dinco? an not do mermy. han

Yoll Vyingrs in welve seak sher,
I shince your mine:
thare, not with Math. US
